<a href="https://colab.research.google.com/github/leoalfonso/M15-mcdm-dashboard/blob/main/MCA_SbN_SAW_TOPSIS_VIKOR_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%"><tr><td style="width:70%;text-align:center"><h1>Análisis multicriterio para decisiones sobre el agua</h1><h2>SAW, TOPSIS y VIKOR</h2>Por: Leonardo Alfonso</td><td><img width="220" src="https://ihe-delft-ihe-website-production.s3.eu-central-1.amazonaws.com/s3fs-public/styles/792w/public/2022-11/IHE-DELFT-INSTITUTE_UNESCO_RGB.png?itok=-GnfBc2x"></td></tr></table>

# A. Introducción
## Objetivos de aprendizaje
Construir e interpretar una matriz de decisión, distinguir criterios de beneficio y costo, aplicar tres métodos de MCA y explicar cómo cambian sus resultados al variar las preferencias.

## Problema
¿Cómo comparar alternativas de gestión del agua cuando el costo, los ecosistemas y la aceptación social entran en conflicto? El **análisis multicriterio (MCA o AMC)** estructura esa comparación. **MCDM/MCDA** se refiere a la toma de decisiones/análisis multicriterio. **MADM** compara un conjunto finito de alternativas, como aquí. La optimización multiobjetivo genera soluciones a partir de variables y restricciones; después puede aplicarse MCA para elegir entre ellas.

Se utiliza para **selección**, **ordenación**, **clasificación en categorías** o descripción de opciones. Este cuaderno trata la ordenación. En SbN permite comparar restauración de humedales, reforestación ribereña o medidas híbridas, siempre con alternativas técnicamente viables.

- **Alternativa**: opción evaluada, fila $i$.
- **Objetivo**: resultado deseado, por ejemplo reducir inundaciones.
- **Criterio/indicador**: dimensión y medida operacional, columna $j$.
- **Desempeño** $x_{ij}$: valor de la alternativa $i$ en el criterio $j$.
- **Sentido**: beneficio (+1, maximizar) o costo (−1, minimizar).
- **Peso** $w_j\geq0$, $\sum_jw_j=1$: preferencia relativa, interpretada junto con la escala de normalización.
- **Puntuación**: salida del método. **Rango**: posición ordinal, 1 es la mejor; admite empates.

La entrada es $X=[x_{ij}]_{m\times n}$, acompañada de pesos y sentidos. Las restricciones obligatorias se comprueban **antes** de agregar criterios: un beneficio alto no debe compensar un incumplimiento legal. Evitar criterios redundantes y documentar quién asigna los pesos.

## Material, formato del CSV y uso
Puede **subir su propio CSV** al ejecutar la celda de carga en Colab. No se exige un nombre de archivo ni nombres específicos de columnas, ni tres alternativas o cinco criterios.

**Formato válido:** primera fila con nombres únicos de columnas; por defecto la primera columna contiene identificadores únicos de alternativas y las restantes son criterios numéricos. Al menos dos alternativas y un criterio. No incluir filas de pesos, sentidos, unidades o totales dentro de la matriz. Se permiten ceros y números negativos, pero no celdas vacías, texto en los criterios, NaN o infinitos. No usar separadores de miles.

Ejemplo:
```csv
Alternativa,Inversion,Biodiversidad,Aceptacion
Humedal,120,8,7
Bosque,95,6,9
Solucion_hibrida,150,9,8
```

En la celda de carga se puede elegir separador (automático, coma, punto y coma o tabulador), símbolo decimal y codificación. Para CSV de Excel con decimales con coma: separador `;` y decimal `,`. Si todas las columnas son criterios numéricos, desactiva **PRIMERA_COLUMNA_ES_ID**: se generarán nombres A1, A2, etc.

**Pasos:** ejecutar instalación e importaciones; subir el CSV; ejecutar configuración y responder las preguntas sobre sentido y peso de cada criterio; continuar con los métodos. Los pesos se normalizan automáticamente. Ningún nombre de columna determina su sentido. Para cambiar de archivo, repetir desde la carga y volver a ejecutar las celdas posteriores. Los resultados antiguos se han eliminado de este archivo para no confundirlos con tu matriz.

El `data.csv` original sigue disponible como ejemplo al activar **USAR_EJEMPLO**. En ese caso se sugieren sus sentidos originales (costo a minimizar), que se debe revisar. A1–A3 son identificadores genéricos; las unidades originales no están documentadas.


# B. La biblioteca pymcdm
[pymcdm](https://pymcdm.readthedocs.io/en/v1.3.1/) reúne métodos multicriterio, normalizaciones, ponderaciones y herramientas de comparación. Aquí usamos versión **1.3.1**.

### Paso 1. Instalar e importar
La primera celda instala la versión utilizada. Las siguientes permiten subir el CSV, validar los datos y configurar las preferencias. Las preguntas aceptan punto o coma decimal para los pesos.

In [ ]:
%pip -q install pymcdm==1.3.1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown
from pymcdm.methods import WSM, TOPSIS, VIKOR
from pymcdm.normalizations import minmax_normalization
from scipy.stats import rankdata
from importlib.metadata import version
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
print('pymcdm:', version('pymcdm'))
SAW = WSM


In [ ]:
#@title Paso 2a. Subir una matriz CSV
import csv
import io

USAR_EJEMPLO = False #@param {type:"boolean"}
PRIMERA_COLUMNA_ES_ID = True #@param {type:"boolean"}
SEPARADOR = ";" #@param ["automatico", ",", ";", "tabulador"]
DECIMAL = "." #@param [".", ","]
CODIFICACION = "utf-8-sig" #@param ["utf-8-sig", "latin-1"]
CSV_ORIGINAL = "alternative,discharge,cost,wetlands,forest,social acceptance\nA1,2.5,50,0.9,0.1,0.17\nA2,3,60,0.6,0.6,0.83\nA3,4,80,0.1,0.3,0.5\n"

# Evitar reutilizar resultados/configuración de una matriz anterior.
for nombre in ['df','X','Z','pesos','tipos','criterios','alternativas','u','closeness','q','rangos','resultados','sens']:
    globals().pop(nombre, None)

def leer_matriz_csv(contenido, primera_id=True, separador='automatico', decimal='.', codificacion='utf-8-sig'):
    """Validación estructural antes de convertir criterios a números."""
    try:
        texto = contenido.decode(codificacion)
    except UnicodeDecodeError as exc:
        raise ValueError('No se pudo leer la codificación. Prueba latin-1 o guarda el CSV en UTF-8.') from exc
    if not texto.strip(): raise ValueError('El archivo está vacío.')
    if separador == 'automatico':
        try: sep = csv.Sniffer().sniff(texto[:65536], delimiters=',;\t').delimiter
        except csv.Error:
            # Una única columna es válida si todas las columnas son criterios.
            if not primera_id and not any(ch in texto.splitlines()[0] for ch in ',;\t'): sep = ';'
            else: raise ValueError('No se pudo detectar el separador. Selecciónalo manualmente.')
    else: sep = '\t' if separador == 'tabulador' else separador
    if sep == decimal:
        raise ValueError('El separador de columnas debe ser distinto del símbolo decimal.')
    try: filas = [f for f in csv.reader(io.StringIO(texto), delimiter=sep, strict=True) if f]
    except csv.Error as exc: raise ValueError('CSV mal formado: revisa comillas y separadores.') from exc
    cabecera = [x.strip() for x in filas[0]]
    if not all(cabecera) or len(set(cabecera)) != len(cabecera):
        raise ValueError('Los nombres de las columnas deben ser únicos y no vacíos.')
    if len(cabecera) < (2 if primera_id else 1):
        raise ValueError('Falta al menos una columna de criterios numéricos.')
    if len(filas) < 3: raise ValueError('Se requieren al menos dos alternativas.')
    for fila_num, fila in enumerate(filas[1:], start=2):
        if len(fila)!=len(cabecera):
            raise ValueError(f'Fila {fila_num}: tiene {len(fila)} campos; se esperaban {len(cabecera)}.')
    bruto = pd.DataFrame(filas[1:], columns=cabecera).map(lambda x: x.strip())
    nombres = bruto.iloc[:,0] if primera_id else pd.Series([f'A{i+1}' for i in range(len(bruto))])
    if (nombres=='').any() or nombres.duplicated().any():
        raise ValueError('Los identificadores de alternativas deben ser únicos y no vacíos.')
    datos = bruto.iloc[:,1:] if primera_id else bruto
    convertido = pd.DataFrame(index=datos.index)
    for col in datos:
        valores = datos[col].str.replace(decimal, '.', regex=False) if decimal != '.' else datos[col]
        numericos = pd.to_numeric(valores, errors='coerce')
        invalidos = ~np.isfinite(numericos.to_numpy(float))
        if invalidos.any():
            filas_error = (np.flatnonzero(invalidos)[:8]+2).tolist()
            raise ValueError(f'Criterio «{col}»: valores vacíos, no numéricos o infinitos en filas {filas_error}.')
        convertido[col] = numericos.astype(float)
    convertido.index = pd.Index(nombres.to_numpy(), name='Alternativa')
    return convertido

if USAR_EJEMPLO:
    contenido_csv = CSV_ORIGINAL.encode('utf-8')
    nombre_csv = 'Ejemplo original integrado'
    df = leer_matriz_csv(contenido_csv, True, ',', '.', 'utf-8-sig')
else:
    try:
        from google.colab import files
    except ImportError:
        ruta_local = input('Ruta local del archivo .csv: ').strip()
        if Path(ruta_local).suffix.lower() != '.csv': raise ValueError('Selecciona un archivo .csv.')
        contenido_csv = Path(ruta_local).read_bytes(); nombre_csv = Path(ruta_local).name
    else:
        subidos = files.upload()
        if len(subidos)!=1: raise ValueError('Sube exactamente un archivo CSV y vuelve a ejecutar esta celda.')
        nombre_csv, contenido_csv = next(iter(subidos.items()))
        if not nombre_csv.lower().endswith('.csv'): raise ValueError('El archivo debe tener extensión .csv.')
    df = leer_matriz_csv(contenido_csv, PRIMERA_COLUMNA_ES_ID, SEPARADOR, DECIMAL, CODIFICACION)
print(f'Archivo: {nombre_csv}. Matriz válida: {len(df)} alternativas × {len(df.columns)} criterios.')
display(df)


In [ ]:
# Paso 2b. Configurar sentidos y pesos. No se deducen del nombre del criterio.
if 'df' not in globals(): raise ValueError('Primero sube y valida una matriz en la celda anterior.')
criterios = df.columns.tolist()
alternativas = df.index.to_numpy(str)
X = df.to_numpy(float)
etiquetas = criterios.copy()

def pedir_configuracion(nombres, ejemplo=False):
    pesos_lista, tipos_lista = [], []
    print('Sentido: + para maximizar, - para minimizar. Peso: número no negativo; se normalizará.')
    for nombre in nombres:
        defecto = '-' if ejemplo and nombre == 'cost' else '+'
        while True:
            sentido = input(f'{nombre} | sentido (+/-) [{defecto}]: ').strip() or defecto
            if sentido in ['+', '+1', '1', '-', '-1']: break
            print('Escribe + o -.')
        while True:
            try:
                peso = float((input(f'{nombre} | peso relativo [1]: ').strip() or '1').replace(',', '.'))
                if not np.isfinite(peso) or peso < 0: raise ValueError
                break
            except ValueError: print('El peso debe ser un número finito mayor o igual a cero.')
        tipos_lista.append(-1 if sentido in ['-', '-1'] else 1); pesos_lista.append(peso)
    w = np.array(pesos_lista, float)
    if w.max()==0: raise ValueError('Al menos un peso debe ser positivo. Repite esta celda.')
    w = w/w.max(); w = w/w.sum()
    return w, np.array(tipos_lista)

pesos_solicitados, tipos = pedir_configuracion(criterios, USAR_EJEMPLO)
# Escalado previo por columna para evitar desbordamiento con números muy grandes.
escala = np.max(np.abs(X), axis=0); escala[escala==0] = 1
X_escalada = X/escala
constantes = np.ptp(X_escalada, axis=0)==0
activos = (~constantes) & (pesos_solicitados>0)
pesos = np.where(activos, pesos_solicitados, 0.0)
if pesos.sum()>0: pesos /= pesos.sum()
if constantes.any(): print('Criterios constantes omitidos:', ', '.join(np.array(criterios)[constantes]))
if not activos.any(): print('Sin criterios discriminantes con peso positivo: todas las alternativas empatan.')
display(pd.DataFrame({'Criterio':criterios,'Peso solicitado':pesos_solicitados,'Peso efectivo':pesos,
                      'Sentido':np.where(tipos==1,'Maximizar','Minimizar'), 'Constante':constantes}))
Z = np.zeros_like(X_escalada)
for j in np.flatnonzero(~constantes):
    Z[:,j] = minmax_normalization(X_escalada[:,j], cost=tipos[j]==-1)

# Funciones comunes para la matriz base y para la sensibilidad.
saw = SAW(normalization_function=minmax_normalization)
topsis = TOPSIS(normalization_function=minmax_normalization)
def evaluar_metodos(w, v=.5):
    activos_w = (w>0) & (~constantes)
    if not activos_w.any():
        return np.zeros(len(X)), np.full(len(X),.5), np.full(len(X),np.nan)
    ww = w[activos_w]/w[activos_w].sum()
    # Z ya está orientada: todos los criterios activos son de beneficio.
    matriz = Z[:,activos_w]; sentidos = np.ones(activos_w.sum(),dtype=int)
    uu = saw(matriz,ww,sentidos); cc = topsis(matriz,ww,sentidos)
    ss=((1-matriz)*ww).sum(axis=1); rr=((1-matriz)*ww).max(axis=1)
    qq = np.full(len(X),np.nan)
    if np.ptp(ss)>1e-12 and np.ptp(rr)>1e-12:
        qq = VIKOR(v=v)(matriz,ww,sentidos)
    return uu,cc,qq

# Para matrices grandes se limita solo la vista; se evalúan todas las filas y columnas.
filas_vista=min(len(X),30); cols_vista=min(len(criterios),20)
fig,ax=plt.subplots(figsize=(max(8,.65*cols_vista),max(3,.3*filas_vista)))
im=ax.imshow(Z[:filas_vista,:cols_vista],vmin=0,vmax=1,cmap='YlGnBu',aspect='auto')
ax.set(xticks=range(cols_vista),xticklabels=etiquetas[:cols_vista],yticks=range(filas_vista),yticklabels=alternativas[:filas_vista])
plt.setp(ax.get_xticklabels(),rotation=45,ha='right')
if filas_vista*cols_vista<=150:
    for i in range(filas_vista):
        for j in range(cols_vista): ax.text(j,i,f'{Z[i,j]:.2f}',ha='center',va='center',color='white' if Z[i,j]>.6 else 'black')
fig.colorbar(im,ax=ax,label='Desempeño normalizado (1 = mejor; constantes = 0)')
plt.tight_layout();plt.show()
if filas_vista<len(X) or cols_vista<len(criterios): print('Vista parcial de la matriz. Los cálculos usan todos los datos.')
VISTA = np.arange(min(len(X),20))


# C. Métodos y aplicación paso a paso
## SAW: suma ponderada
Normalizar cada criterio para que 1 sea mejor y 0 peor. Para beneficio, $z_{ij}=(x_{ij}-\min x_j)/(\max x_j-\min x_j)$; para costo, $z_{ij}=(\max x_j-x_{ij})/(\max x_j-\min x_j)$.

$U_i=\sum_jw_jz_{ij}$. **Mayor es mejor.** Es transparente y compensatorio: un buen valor puede compensar otro malo. La escala y los pesos determinan esa compensación.

## TOPSIS: cercanía relativa al ideal
Con $y_{ij}=w_jz_{ij}$, el ideal positivo toma el mejor valor de cada columna y el negativo el peor. Calcular distancias euclidianas $D_i^+$ y $D_i^-$ y $C_i=D_i^-/(D_i^++D_i^-)$. **Mayor es mejor.** El ideal puede ser una combinación inexistente. Aquí usamos min–max, no la normalización vectorial de otras formulaciones.

## VIKOR: solución de compromiso
La brecha orientada es $d_{ij}=(f_j^*-x_{ij})/(f_j^*-f_j^-)$, donde $f_j^*$ es el mejor y $f_j^-$ el peor según el sentido. $S_i=\sum_jw_jd_{ij}$ mide brecha agregada y $R_i=\max_j(w_jd_{ij})$ la peor brecha ponderada.

$Q_i=v(S_i-S^*)/(S^- -S^*)+(1-v)(R_i-R^*)/(R^- -R^*)$.

Los asteriscos indican mínimos de S y R, y los superíndices menos sus máximos. **Menor Q es mejor.** $v=0.5$ equilibra el componente agregado y la peor brecha. No representa un peso de criterio ni garantiza equidad social.

Para proponer una única solución de compromiso se comprueba **ventaja aceptable** $Q(a_2)-Q(a_1)\geq1/(m-1)$ y **estabilidad** (a₁ también es mejor en S o R). Si fallan, se propone un conjunto. Denominadores nulos requieren tratar el caso degenerado explícitamente.

Ejecutar ahora las tres celdas, una por método. Cada una muestra su explicación, resultados y gráfico.

In [ ]:
display(Markdown('### Paso 3. SAW: sumar las contribuciones ponderadas\nCada segmento muestra cuánto aporta un criterio. Una barra más larga es mejor.'))
u, closeness, q = evaluar_metodos(pesos)
assert np.allclose(u, Z @ pesos)
display(pd.DataFrame({'SAW':u, 'Rango':rankdata(-u)}, index=alternativas))
fig,ax=plt.subplots(figsize=(10,max(4,.35*len(VISTA)))); base=np.zeros(len(VISTA))
for j in range(X.shape[1]):
    valores=Z[VISTA,j]*pesos[j]
    ax.barh(alternativas[VISTA],valores,left=base,label=etiquetas[j]); base+=valores
ax.set_xlabel('SAW: mayor es mejor'); ax.legend(loc='upper center',bbox_to_anchor=(.5,-.2),ncol=3)
plt.tight_layout(); plt.show()


In [ ]:
display(Markdown('### Paso 4. TOPSIS: acercarse al ideal y alejarse del antiideal\nComparar las dos distancias y la cercanía relativa C.'))
_, closeness, _ = evaluar_metodos(pesos)
Y=Z*pesos
Dmas=np.linalg.norm(Y-Y.max(axis=0),axis=1)
Dmenos=np.linalg.norm(Y-Y.min(axis=0),axis=1)
if activos.any(): assert np.allclose(closeness,Dmenos/(Dmas+Dmenos))
display(pd.DataFrame({'D+':Dmas,'D−':Dmenos,'TOPSIS':closeness,'Rango':rankdata(-closeness)},index=alternativas))
fig,axs=plt.subplots(1,2,figsize=(10,4))
axs[0].scatter(Dmas,Dmenos,s=100)
for i in VISTA:
    a=alternativas[i]; axs[0].annotate(a,(Dmas[i],Dmenos[i]),xytext=(7,5),textcoords='offset points')
axs[0].set(xlabel='D+ (menor es mejor)',ylabel='D− (mayor es mejor)')
axs[1].bar(alternativas[VISTA],closeness[VISTA],color='#00a6d6');axs[1].set(ylabel='TOPSIS C (mayor es mejor)',ylim=(0,1))
axs[1].tick_params(axis='x',labelrotation=60)
plt.tight_layout();plt.show()


In [ ]:
display(Markdown('### Paso 5. VIKOR: ordenar y comprobar el compromiso\nS, R y Q se minimizan. No basta con elegir el Q menor.'))
v=.5
S=((1-Z)*pesos).sum(axis=1); R=((1-Z)*pesos).max(axis=1)
_, _, q = evaluar_metodos(pesos,v)
vikor_disponible = np.isfinite(q).all()
display(pd.DataFrame({'S':S,'R':R,'Q':q,'Rango':rankdata(q) if vikor_disponible else np.full(len(X),np.nan)},index=alternativas))
if not vikor_disponible:
    print('VIKOR no definido: S o R tiene rango nulo. No se propone un compromiso VIKOR. SAW y TOPSIS siguen disponibles.')
else:
    assert np.allclose(q,v*(S-S.min())/np.ptp(S)+(1-v)*(R-R.min())/np.ptp(R))
    orden=np.argsort(q,kind='stable');a1,a2=orden[:2];DQ=1/(len(X)-1)
    ventaja=q[a2]-q[a1]>=DQ-1e-10
    estable=np.isclose(S[a1],S.min()) or np.isclose(R[a1],R.min())
    if ventaja and estable: compromiso=[a1]
    elif not ventaja:
        compromiso=[i for i in orden if q[i]-q[a1]<DQ-1e-10]
        if not estable: compromiso=list(dict.fromkeys(compromiso+[a2]))
    else: compromiso=[a1,a2]
    print(f'Ventaja aceptable: {ventaja} (umbral {DQ:.3f}); estabilidad: {estable}')
    print('Solución/conjunto de compromiso:', ', '.join(alternativas[compromiso]))
fig,axs=plt.subplots(1,3,figsize=(11,3.5))
for ax,val,nombre in zip(axs,[S,R,q],['S','R','Q']):
    ax.bar(alternativas[VISTA],val[VISTA],color='#548235');ax.set_ylabel(nombre+' (menor es mejor)');ax.tick_params(axis='x',labelrotation=60)
    if nombre=='Q' and not vikor_disponible: ax.text(.5,.5,'No definido',transform=ax.transAxes,ha='center')
plt.tight_layout();plt.show()


# D. Comparación, sensibilidad e interpretación
## Paso 6. Comparar posiciones
Los gráficos individuales muestran hasta 20 alternativas; el mapa de calor, hasta 30 filas y 20 criterios. Las tablas y el archivo `resultados_mca.csv` incluyen **todas** las alternativas. NaN en VIKOR significa «no definido», no una alternativa peor.
Comparar **rangos**, no los tamaños de SAW, C y Q. Las discrepancias reflejan reglas de agregación diferentes. Un empate conserva su rango medio; no convertirlo en un entero arbitrario.

## Paso 7. Explorar preferencias
Elegir un criterio no constante y variar su peso manteniendo proporcionales los restantes y variar `v` en VIKOR. Esto es sensibilidad unidimensional, no una demostración de robustez ante toda la incertidumbre. Los pesos iguales del escenario base no equivalen a neutralidad.

## Preguntas de discusión
1. ¿Qué alternativa resulta preferida con pesos iguales? ¿Se cumplen ambas condiciones de VIKOR?
2. ¿Qué sucede al aumentar el peso de un criterio seleccionado? ¿En qué intervalos cambia el primer puesto?
3. Si el caudal fuera un vertido no deseado, ¿qué sentido usarías? Repite la celda de configuración y ejecuta las celdas posteriores.
4. Para una aplicación SbN, ¿qué unidades, umbrales ecológicos, grupos sociales y datos de incertidumbre faltan?

**Conclusión:** el ranking es condicional a los datos, escalas, preferencias y método. Debe apoyar una deliberación documentada. Los datos A1–A3 no reproducen el caso A13 de la presentación.

## Fuentes
- [Documentación pymcdm 1.3.1](https://pymcdm.readthedocs.io/en/v1.3.1/modules/american_school.html).
- Kizielewicz, Shekhovtsov y Sałabun (2023). *pymcdm—The universal library for solving multi-criteria decision-making problems*. SoftwareX 22, 101368. https://doi.org/10.1016/j.softx.2023.101368
- Opricovic y Tzeng (2004). *Compromise solution by MCDM methods: A comparative analysis of VIKOR and TOPSIS*. EJOR 156, 445–455. https://doi.org/10.1016/S0377-2217(03)00020-1



In [ ]:
resultados=pd.DataFrame({'SAW':u,'TOPSIS':closeness,'VIKOR Q':q},index=alternativas)
rangos=pd.DataFrame({'SAW':rankdata(-u),'TOPSIS':rankdata(-closeness),'VIKOR':rankdata(q) if np.isfinite(q).all() else np.full(len(X),np.nan)},index=alternativas)
display(resultados.round(4));display(rangos)
fig,ax=plt.subplots(figsize=(7,4))
for a in alternativas[VISTA]: ax.plot(rangos.columns,rangos.loc[a],marker='o',label=a)
ax.set(yticks=np.unique(np.linspace(1,len(X),min(len(X),10)).astype(int)),ylabel='Posición (1 = mejor)');ax.invert_yaxis();ax.legend();ax.grid(alpha=.25)
plt.tight_layout();plt.show()
resultados.join(rangos.add_prefix('rango_')).to_csv('resultados_mca.csv',index_label='Alternativa')


In [ ]:
# Paso 7. Sensibilidad para un criterio elegido por su nombre.
validos=[j for j in range(len(criterios)) if not constantes[j]]
if len(validos)<2 or not activos.any():
    print('Sensibilidad de pesos omitida: se necesitan al menos dos criterios no constantes y peso efectivo positivo.')
else:
    print('Criterios disponibles:', ', '.join(criterios[j] for j in validos))
    while True:
        elegido=input(f'Criterio para sensibilidad [{criterios[validos[0]]}]: ').strip() or criterios[validos[0]]
        if elegido in [criterios[j] for j in validos]: break
        print('Escribe exactamente uno de los nombres indicados.')
    k=criterios.index(elegido); otros=(~constantes)&(np.arange(len(pesos))!=k)
    proporcion=pesos[otros].copy()
    if proporcion.sum()==0:
        proporcion=np.ones(otros.sum())
        print('Los demás pesos eran cero: en esta exploración se repartirán por igual.')
    proporcion/=proporcion.sum()
    grid=np.linspace(.05,.95,41);registros=[]
    for wc in grid:
        w=np.zeros_like(pesos);w[otros]=proporcion*(1-wc);w[k]=wc
        uu,cc,qq=evaluar_metodos(w,v)
        for metodo,val in {'SAW':-uu,'TOPSIS':-cc,'VIKOR':qq}.items():
            rank=rankdata(val) if np.isfinite(val).all() else np.full(len(X),np.nan)
            for a,r in zip(alternativas,rank): registros.append([wc,metodo,a,r])
    sens=pd.DataFrame(registros,columns=['Peso','Método','Alternativa','Rango'])
    sens.to_csv('sensibilidad_pesos.csv',index=False)
    fig,axs=plt.subplots(1,3,figsize=(12,4),sharey=True)
    for ax,met in zip(axs,['SAW','TOPSIS','VIKOR']):
        for a in alternativas[VISTA]:
            d=sens[(sens['Método']==met)&(sens['Alternativa']==a)]
            ax.plot(d['Peso'],d['Rango'],label=a)
        ax.set(title=met,xlabel=f'Peso de {elegido}');ax.grid(alpha=.2)
    axs[0].set_ylabel('Posición (1 = mejor)');axs[0].invert_yaxis();axs[-1].legend(bbox_to_anchor=(1.04,1))
    plt.tight_layout();plt.show()
    print('Los huecos en VIKOR indican Q no definido para esos pesos.')
if vikor_disponible:
    v_grid=np.linspace(0,1,21)
    q_grid=np.array([evaluar_metodos(pesos,float(vv))[2] for vv in v_grid])
    fig,ax=plt.subplots(figsize=(8,4))
    for i in VISTA: ax.plot(v_grid,q_grid[:,i],label=alternativas[i])
    ax.set(xlabel='v: mayor énfasis en la brecha agregada S',ylabel='Q (menor es mejor)');ax.legend(bbox_to_anchor=(1.04,1));ax.grid(alpha=.2)
    plt.tight_layout();plt.show()
    print('Al cambiar v, comprobar de nuevo las condiciones de aceptación antes de recomendar una opción.')
else:
    print('Sensibilidad de v omitida porque VIKOR no está definido en el escenario base.')
